# 零、总述

随着学习的深入，我们开始使用主要针对**文本生成**进行训练的模型，这类模型通常被称为**生成式预训练的Transformer(Generativ Pre-trained Transformer, GPT)。** 这些模型具有非常强大的能力，可以根据用户提供的提示词生成文本。通过**提示词工程（Prompt Engineering）**，我们可以更合理的设计这些提示词，从而提高模型生成文本的质量。

在本章中，我们将更加深入地探索这些生成式模型，并进一步学习**提示词工程、使用生成式模型进行推理、结果验证，以及对模型输出进行评估** 等内容。

# 一、使用文本生成模型

在开始学习**提示词工程**的基础知识之前，我们首先需要了解如何使用一个文本生成模型。我们应该怎样选择要使用的模型呢？是选择闭源模型（Proprietary）还是开源模型？

这些问题将作为我们使用文本生成模型的起点

## 1.1 选择 Text Generation Model

我们以选择闭源和开源模型来开始选择文本生成模型，即使专有模型的性能会很好，但是为了学习使用，我们使用开源模型。

那么，我们使用 `Phi-3-mini` 模型，它有 3.8B（38亿）的参数量，特别适合在 8G 的 VRAM 上运行。

总的来说，将小模型变成大模型要比从大模型变成小模型要容易得多。更小的模型会提供更好的引导，而且可以为后续大模型的学习会打下坚实的基础

## 1.2 加载文本生成模型

In [ ]:
import torch
from sympy.polys.polyconfig import query
from torch.distributed.autograd import context
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 加载模型的分词器
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi3-mini-4k-instruct")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

还记得我们第一个notebook中的示例么，我们让他讲一个笑话，我们依旧使用这个例子

In [ ]:
# 提示词
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])

在内部，`transformers.pipeline` 首先会将我们的 messages 转换成特定的提示词模板，我们可以使用下述的函数看一下模型将我们的 messages 转成了什么：

In [ ]:
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)

在输出结果中，你可以看到有两个特殊的词元：`<|user|>` 和 `<|assistant|>`，这个提示词模版的说明，如下图所示

<center>
<img src="./resources/phi3-template.png">
</center>

## 1.3 控制模型输出

除了提示词工程之外，我们可以调整模型的参数，比如：`temperature` 和 `top_p`.

这些参数会控制输出的随机性。LLM 每次生成一个 token 时，每一个可能的 token 都会被分配一个似然值。

当我们加载模型时，设置 `do_sample=False` 的目的是为了确保它生成的内容有一些连贯性，它意味着每次生成的 token 都是最可能的那一个词。然而，为了使用 `temperature` 和 `top_p` 参数，我们将设置 `do_sample=True`。

1. temperature

`temperature` 参数控制文本生成的随机性和多样性。它定义了有多大的可能会选择小概率的 token。更高的 `temperature` 通常会导致更*多样化*的输出，而更低的 `temperature` 会生成更稳定的输出。请注意，即使你设置了相同的温度值，多次运行下述代码结果也是会变化的，因为 `temperature` 引入了随机选择的行为

In [ ]:
# 更高的温度
output1 = pipe(messages, do_sample=True, temperature=0.8)
print(output1[0]["generated_text"])

In [ ]:
# 更低的温度
output2 = pipe(messages, do_sample=True, temperature=0.2)
print(output2[0]["generated_text"])

2. top-p

`top-p`：也叫做**核采样**，它是一种采样的技术，来控制 LLM 来选择哪些 token 的子集，这些 token 的子集就被称为 nucleus(核)。它会选择那些直到累积的概率达到其设定的值的 token，所以，如果将 `top_p` 设置为 1，则它会选择所有 token。`top_p` 对模型生成的 token 的影响如下：

<center>
<img src="./resources/top_p.png">
</center>


In [ ]:
output = pipe(messages, do_sample=True, top_p=0.6)
print(output[0]["generated_text"])

# 二、介绍提示词工程

**提示词工程（Prompt Engineering）** 是我们可以精心设计提示词，可以引导 LLM 生成我们期望的回答，**但是， 提示词工程不仅仅是设计有效的提示词**，它还可以作为一种工具，用来评估模型的输出，以及设计防护机制和安全环节措施。不存在一个完美的提示词设定，未来也可能不会存在。

我们从回答一个问题开始吧：在提示词中应该有什么呢？

## 2.1 提示词的基本成分


LLM 本质上是一台预测机器，我们从基础的示例看起，如下图所示，可以看到由于没有给出明确的指令，因此 LLM 只会尝试续写这个句子。

```text
                         Basic prompt
                 ┌─────────────────────┐
Input ─────────> │    The sky is       │
                 └─────────────────────┘
                           │
                           ▼
                      ┌─────────┐
                      │   LLM   │
                      └─────────┘
                           │
                           ▼
                         Output
                           │
                           ▼
Generated text ─────────> blue.
```

更进一步来看，在提示词工程中，*我们通常会通过提出一个明确的问题，或者指定一个需要 LLM 完成的具体任务来进行设计。* 为了让模型生成我们更期望的回答，我们需要一个结构更加清晰的提示词。如下图所示，可以看到它包含两个组成部分：**指令本身**以及 **与该指令相关的数据**。

```text
                     Instruction prompt

Instruction ──────>  ┌──────────────────────────────────────────────┐
                     │ Classify the text into negative or positive. │
Data ─────────────>  │                                              │
                     │        "This is a great movie!"              │
                     └──────────────────────────────────────────────┘
                                      │
                                      ▼
                                 ┌─────────┐
                                 │   LLM   │
                                 └─────────┘
                                      │
                                      ▼
                                    Output

Generated text ──────>  The text is positive.
```

再进一步来看，我们在上面的基础上添加了 "Text" 和 "Sentiment"，以防止模型生成一个完整的句子。由于 LLM 已经接收到了足够的指令，因此它有能力泛化到我们所提供的结构中，所以我们期望它会输出 `negaitve` 或 `positive`。

```text
                 Instruction prompt
                 With output indicator

Instruction ──────>  ┌──────────────────────────────────────────────┐
                     │ Classify the text into negative or positive. │
                     │                                              │
Output indicators ─> │ Text:      "This is a great movie!"          │ <──── Data
                     │                                              │
                     │ Sentiment:                                   │
                     └──────────────────────────────────────────────┘
                                      │
                                      ▼
                                 ┌─────────┐
                                 │   LLM   │
                                 └─────────┘
                                      │
                                      ▼
                                    Output

Generated text ──────>  Positive
```

我们可以不断增加和调整提示词中的不同元素，直至模型生成我们所期望的回答。

## 2.2 基于指令的提示词

虽然提示词有很多种方式：从讨论哲学到角色扮演，但是往往提示词会被用来回答特定的问题或者完成一项特定的任务，这被称为基于指令的提示词。下图是基于指令的提示词的使用实例：

<center>
<img src="./resources/instruction_based_prompt_use_case.png">
</center>

上面图中的每一个任务都需要不同的提示词格式，不同的指向和不同的发问。下图是几个使用实例的具体例子：

<center>
<img src="./resources/instruction_based_prompt_example.png">
</center>

虽然这些任务需要不同的指令，但是在提示词技术中也有一些重叠的地方来对输出进行改进。这些技术包含但不限于：

1. Specificity: 指向性，明确性

精确地描述你想要完成的。不是去对 LLM 说“给我写一个产品的描述”，而是要说**给我写一个少于两句话的产品描述，要用正式的用语**

2. Hallucination: 幻觉

LLM 可能会堂而皇之地生成不正确的信息，这被称为幻觉。为了减少这个影响，如果 LLM 直到问题的答案的话，我们只让他生成一个。如果他不知道，则让他用 “我不知道” 来代替。

3. Order: 顺序

你的提示词即用指令开始也用指令结束。特别是长提示词，LLM 通常会忽略中间的信息，而去关注开头或者结尾的提示词.

在这三个方面中，可以说**Specificity 是最重要的方面**，通过限制并明确模型应该生成什么内容，可以降低模型生成与实际场景无关内容的概率。就像人与人之间的交流一样，如果没有明确的指令或额外的上下午，就很那判断当前任务究竟要完成什么。


# 三、高级的提示词工程

提示词设计会变得很复杂，接下来，我们将介绍几种用于构建提示词的高级技巧，我们会从迭代式构建复杂提示词的工作流程开始，一直到按顺序使用多个 LLM 来获得更好的结果。最终，我们还会进一步介绍高级推理技术。

## 3.1 提示词潜在的复杂度

一个提示词通常由多个组成部分构成，在最开始的示例中，我们的提示词由**指令、数据和输出指示符组成**，但提示词并不局限于这个三个组成部分，可以根据需要做扩展。

这些高级的组件会让提示词变得相当的复杂。一些常用的组件有：

1. Persona：LLM 应该扮演什么角色
2. Instruction：描述任务本身，尽可能具体
3. Context：用于描述问题或任务背景的额外信息
4. Format：指定 LLM 应该使用什么格式来输出生成的文本
5. Audience：生成文本所面向的目标人群
6. Tone：生成文本的语气
7. Data：数据

下面，我们来扩展之前的分类提示词，并使用前面提到的所有组成部分。我们可以逐步的构建提示词，并探索每一次修改带来的结果。

那我们大致依据下图来进行探索，**注意：不同组件的排列顺序也会影响 LLM 的输出质量，因为其有近因效应(recency effect)和首因效应（primacy effect）**

<center>
<img src="./resources/modular_component_prompt.png">
</center>

In [ ]:
# persona = "你是 LLM 的专家，你特别擅长将复杂的论文内容处理成理解简单的总结"
persona = "You are an expert in Large Language models.You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key finding of the paper provided.\n"
context = "Your summary should extract the most crucial points that can help researches quickly understand the most vital information of the paper.\n"
# data_format = "请撰写一份要点式摘要，概述该方法，随后用一段简练的文字总结主要结果“
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"
tone = "The tone should be professional and clear.\n"

下面这段文本是 DeepSeek-R1 的论文，其中包括 R1-Zero、纯 RL 推理能力涌现、cold-start、多阶段训练、蒸馏到 1.5B–70B 模型 等论文核心信息。

In [ ]:
text = """
DeepSeek-R1 investigates how reinforcement learning can be used to
develop advanced reasoning capabilities in large language models.

The work introduces two reasoning models: DeepSeek-R1-Zero and
DeepSeek-R1. DeepSeek-R1-Zero is trained using large-scale reinforcement
learning without supervised fine-tuning as an initial stage. During
training, the model naturally develops reasoning behaviors such as
self-reflection, verification, and longer reasoning processes. This
demonstrates that sophisticated reasoning abilities can emerge through
reinforcement learning rather than relying entirely on human-written
reasoning examples.

However, DeepSeek-R1-Zero also exhibits several problems, including
poor readability and language mixing. To address these limitations,
DeepSeek-R1 introduces a multi-stage training pipeline that combines
cold-start data, reinforcement learning, and supervised fine-tuning.
The cold-start data helps establish more readable reasoning patterns
before large-scale reinforcement learning is applied.

Experimental evaluations show that DeepSeek-R1 achieves strong
performance on mathematics, coding, and reasoning benchmarks and
reaches performance comparable to leading reasoning models such as
OpenAI o1-1217 on several tasks.

The authors also investigate knowledge distillation. Reasoning patterns
generated by DeepSeek-R1 are transferred to smaller dense models based
on Qwen and Llama architectures. These distilled models range from
1.5B to 70B parameters and demonstrate that reasoning capabilities
learned by a large model can be effectively transferred to smaller
models.

Overall, the study suggests that reinforcement learning can play a
central role in developing reasoning abilities in language models,
while cold-start training and distillation provide practical ways to
improve usability and transfer reasoning capabilities to smaller models.
"""

In [ ]:
data = f"Text to summarize: {text}"

下面我们来探索这四种组合，依旧使用 `Phi-3-mini`模型

In [ ]:
# 1. instruction + data
print("=============== instruction + data ===============")
query1 = instruction + data
messages = [
    {"role": "user", "content": f"{query1}"}
]
# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])


# 2. persona + instruction + data
print("=============== persona + instruction + data ===============")
query2 = persona + instruction + data
messages = [
    {"role": "user", "content": f"{query2}"}
]
# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])

# 3. context + tone + instruction + data
print("=============== context + tone + instruction + data ===============")
query3 = instruction + data
messages = [
    {"role": "user", "content": f"{query3}"}
]
# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])

# 4. all-in
query4 = persona + instruction + context + data_format + audience + tone + data
messages = [
    {"role": "user", "content": f"{query4}"}
]
# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])

## 3.2 上下文内学习：提供例子

在前面的例子中，我们会尝试准确的描述 LLM 应该做什么。*那我们与其描述任务，为什么不直接展示任务呢？* 那么，我们就可以给大模型提供一些示例，这通常被称为 **in-context learning(上下文内学习)**。你可以不给大模型提供示例，也可以给大模型提供一个示例，也可以提供两个或更多的示例。如下图所示

<center>
<img src="./resources/shot_prompt.png">
</cener>


借用非常经典的一句话，“一个例子胜过千言万语”。我们用一个简单的例子来说明这种方法，这个例子来自于最初介绍该方法的论文。

此提示词的目标是生成一个包含虚构词语的句子，所以为了提高生成句子的质量，会给模型展示一个示例。

我们**必须**通过 `user` 和 `assistant` 来区分例子的问题和答案。如果不这样的话，我们似乎是在和我们自己对话。一般情况下，我们使用 `assistant` 代表答案。


In [ ]:
# 虚构词的任务
one_shot_prompt = [
    {
        "role": "user",
        "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
    },
    {
        "role": "assistant",
        "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
    },
    {
        "role": "user",
        # swing a sword at it： 挥剑砍向它
        "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
    }
]

# 分词结果
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))

In [ ]:
output = pipe(one_shot_prompt)
print(output[0]["generated_text"])

## 3.3 提示词链：将问题分解

在前面的示例中，我们探讨了如何将提示词拆分成多个模块化组件，以提升 LLM 的表现，这种方式对于高度复杂的提示词或者任务来说，它可能并不现实。此时我们就可以将一个提示词拆成一条连续的交互链，来分次调用大模型，逐步解决问题。举个例子，假设我们希望让 LLM 根据一系列产品的特征来生成产品名称、宣传口号和销售文案（sales_pitch），那我们就可以首先生成产品名称，然后将产品名称和特征作为输入来生成宣传口号，最后再使用产品特征、产品名称和宣传口号来生成销售文案。

<center>
<img src="./resources/chain_prompt.png">
</center>

In [ ]:
# 创建名称和产品名
product_prompt = [
    {"role": "user", "content": "Create a name and slogan for a chatbot that leverages LLMs."}
]
outputs = pipe(product_prompt)
product_description = outputs[0]["generated_text"]
print(product_description)

In [ ]:
# 创建销售文案
sales_pitch_prompt = [
    {"role": "user", "content": f"Generate a very short sales pitch for the following product: '{product_description}'"}
]

outputs = pipe(sales_pitch_prompt)
print(outputs[0]["generated_text"])

这种方法可以应用于多种使用场景，包括：

1. **Response Validation(响应验证)**

让 LLM 对之前生成的输出进行再次检查

2. **（Parallel Prompts）并行提示**

**并行创建多个提示词，然后在最后一步将它们的结果合并起来。例如，可以让多个 LLM 并行生成多份不同的食谱，然后将这些结果汇总起来生成一份购物清单。**

3. **Writing Stories(故事创作)**

**通过将问题拆分成多个组成部分，利用 LLM 来创作书籍或故事。例如，可以先撰写故事摘要，然后塑造人物角色、构建故事情节节点，最后再进一步创作具体的对话**

# 四、使用生成式模型进行推理

## 4.1 CoT（Chain-of-Thought）: 在回答之前思考

## 4.2 Self-Consistency(自洽性)：样例输出

## 4.3 Tree-of-Thought（思维树）: 探索内部的步骤

# 五、输出验证

## 5.1 提供示例

## 5.2 语法：约束后的样例